In [24]:
#RUN THIS BOX FIRST

import random
#Dictionary goes {Dice identifier : Rolled number on dice, Check if the dice will be rolled, and the Reroll identifier} for each dice
rolls = {"Dice A" : [0, True, "a"], "Dice B" : [0, True, "b"], "Dice C" : [0, True, "c"], "Dice D" : [0, True, "d"], "Dice E" : [0, True, "e"]}
counted_Dice = {"Ones" : 0, "Twos" : 0, "Threes" : 0, "Fours" : 0, "Fives" : 0, "Sixes" : 0}

onesScore = [True, 0]
twosScore = [True, 0]
threesScore = [True, 0]
foursScore = [True, 0]
fivesScore = [True, 0]
sixesScore = [True, 0]
threeOAKScore = [True, 0]
fourOAKScore = [True, 0]
fullHouseScore = [True, 0]
smallStraightScore = [True, 0]
largeStraightScore = [True, 0]
yahtzeeScore = [True, 0]
chanceScore = [True, 0]
extraYahtzees = 0

upperScore = 0
bonusScore = 0
upperTotal = 0
lowerTotal = 0

pairCheck = False
threeCheck = False
fourCheck = False
fullHouseCheck = False
yahtzeeCheck = False

#Lookup tables so one function can score every upper category and one can score every lower category
#{what the player types : (score list, name of the dice in counted_Dice, points per die)}
upper_Categories = {"ones" : (onesScore, "Ones", 1), "twos" : (twosScore, "Twos", 2), "threes" : (threesScore, "Threes", 3),
                    "fours" : (foursScore, "Fours", 4), "fives" : (fivesScore, "Fives", 5), "sixes" : (sixesScore, "Sixes", 6)}
#{what the player types : (score list, name shown to the player)}
lower_Categories = {"three of a kind" : (threeOAKScore, "Three of a Kind"), "four of a kind" : (fourOAKScore, "Four of a Kind"),
                    "full house" : (fullHouseScore, "Full House"), "small straight" : (smallStraightScore, "Small Straight"),
                    "large straight" : (largeStraightScore, "Large Straight"), "yahtzee" : (yahtzeeScore, "Yahtzee"),
                    "chance" : (chanceScore, "Chance")}

def check_Special():
    global pairCheck, threeCheck, fourCheck, fullHouseCheck, yahtzeeCheck
    reset_Special()
    for number, count in counted_Dice.items():
        if count == 2:
            pairCheck = True
        if count >= 3:
            threeCheck = True
        if count >= 4:
            fourCheck = True
        if count == 5:
            yahtzeeCheck = True
    fullHouseCheck = pairCheck and (3 in counted_Dice.values())
def count_Dice():
    reset_Count()
    names = {1 : "Ones", 2 : "Twos", 3 : "Threes", 4 : "Fours", 5 : "Fives", 6 : "Sixes"}
    for diceNum, diceInfo in rolls.items():
        if diceInfo[0] in names:
            counted_Dice[names[diceInfo[0]]] += 1
    print("Ones:", counted_Dice["Ones"], "| Twos:", counted_Dice["Twos"], "| Threes:", counted_Dice["Threes"], "| Fours:", counted_Dice["Fours"], "| Fives:", counted_Dice["Fives"], "| Sixes:", counted_Dice["Sixes"])

def display_Board():     
    print("=" * 31)
    display_Row("Ones", onesScore[1])
    print("-" * 31)
    display_Row("Twos", twosScore[1])
    print("-" * 31)
    display_Row("Threes", threesScore[1])
    print("-" * 31)
    display_Row("Fours", foursScore[1])
    print("-" * 31)
    display_Row("Fives", fivesScore[1])
    print("-" * 31)
    display_Row("Sixes", sixesScore[1])
    print("=" * 31)
    display_Row("Total", upperScore)
    print("-" * 31)
    display_Row("Bonus (" + str(upperScore) + "/63)", bonusScore)
    print("-" * 31)
    display_Row("Upper Total", upperTotal)
    print("=" * 31)
    print("")
    print("=" * 31)
    display_Row("3 of a Kind", threeOAKScore[1])
    print("-" * 31)
    display_Row("4 of a Kind", fourOAKScore[1])
    print("-" * 31)
    display_Row("Full House", fullHouseScore[1])
    print("-" * 31)
    display_Row("Sm. Straight (4)", smallStraightScore[1])
    print("-" * 31)
    display_Row("Lg. Straight (5)", largeStraightScore[1])
    print("-" * 31)
    display_Row("YAHTZEE", yahtzeeScore[1])
    print("-" * 31)
    display_Row("Chance", chanceScore[1])
    print("-" * 31)
    display_Row("YAHTZEE BONUS", extraYahtzees)
    print("=" * 31)
    display_Row("Upper Total", upperTotal)
    print("=" * 31)
    display_Row("Lower Total", lowerTotal)
    print("=" * 31)
    display_Row("Grand Total", upperTotal + lowerTotal)
    print("=" * 31)

def display_Row(label, value):
    print(f"| {label:<18} | {value:>5} |")
    
def do_Turn():
    reset_Dice()
    roll_Dice("")
    count_Dice()
    rollsLeft = 2
    while rollsLeft > 0:
        choice = input("What dice do you want to reroll? (type their letters, like 'bad'),Leave empty if you want to score your roll").lower().strip()
        if choice == "":
            break
        validLetters = ""
        for letter in choice:
            if letter in "abcde":
                validLetters += letter
        if validLetters == "":
            print("Please type letters from a to e, or leave it empty to score these dice")
        roll_Dice(validLetters)
        count_Dice()
        rollsLeft -= 1
    score_Dice()    
        
def initialize_Game():
    new_Game()
    print("Game Started!")

def lower_Points(category):
    values = set()
    for diceNum, diceInfo in rolls.items():
        values.add(diceInfo[0])
    if category == "three of a kind":
        return score_All_Dice() if threeCheck else 0
    if category == "four of a kind":
        return score_All_Dice() if fourCheck else 0
    if category == "full house":
        return 25 if fullHouseCheck else 0
    if category  == "small straight":
        if {1,2,3,4} <= values or {2,3,4,5} <= values or {3,4,5,6} <= values:
            return 30
        return 0
    if category == "large straight":
        if values == {1,2,3,4,5} or values == {2,3,4,5,6}:
            return 40
        return 0
    if category == "yahtzee":
        return 50 if yahtzeeCheck else 0
    if category == "chance":
        return score_All_Dice()
    return 0
    
def new_Game():
    global extraYahtzees
    for scoreList in [onesScore, twosScore, threesScore, foursScore, fivesScore, sixesScore, threeOAKScore, fourOAKScore,
                      fullHouseScore, smallStraightScore, largeStraightScore, yahtzeeScore, chanceScore]:
        scoreList[0] = True
        scoreList[1] = 0
    extraYahtzees = 0
    reset_Count()
    reset_Dice()
    reset_Special()
    update_Totals()

def normalize_Category(text):
    text = text.strip().lower()
    if "sm" in text or "small" in text:
        return "small straight"
    if "lg" in text or "large" in text:
        return "large straight"
    if "full" in text:
        return "full house"
    if "kind" in text or "oak" in text:
        if "3" in text or "three" in text:
            return "three of a kind"
        if "4" in text or "four" in text:
            return "four of a kind"
    return text
    
def reset_Count():
    for diceNum, count in counted_Dice.items():
        counted_Dice[diceNum] = 0

def reset_Dice():
    for diceNum,info in rolls.items():
        info[1] = True

def reset_Special():
    global pairCheck, threeCheck, fourCheck, yahtzeeCheck
    pairCheck = False
    threeCheck = False
    fourCheck = False
    yahtzeeCheck = False

def roll_Dice(RerolledDice):
    for diceNum, diceInfo in rolls.items():
        if diceInfo[2] in RerolledDice:
            diceInfo[1] = True
        if diceInfo[1] == True:
            diceInfo[0] = random.randint(1,6)
            diceInfo[1] = False
    for dice, roll in rolls.items():
        print(dice, "rolled a", roll[0])
    return(rolls)

def score_All_Dice():
    total = 0
    for number, count in counted_Dice.items():
        for diceNum, diceInfo in rolls.items():
            total += diceInfo[0]
    return total

def score_Dice():
    global extraYahtzees
    check_Special()
    if yahtzeeCheck and yahtzeeScore[0] == False and yahtzeeScore[1] == 50:
        extraYahtzees += 1
        print("You got the Yahtzee Bonus, scoring you 100 points! Please pick another category for this roll.")
    update_Totals()
    display_Board()
    while True:
        category = normalize_Category(input("What category would you like to score?"))
        if category in upper_Categories:
            done = score_Upper(category)
        elif category in lower_Categories:
            done = score_Lower(category)
        else:
            print("Invalid score category, please try again.")
            done = False
        if done:
            break
    update_Totals()
    display_Board()
                                          
def score_Lower(category):            
    check_Special()
    scoreList, displayName = lower_Categories[category]
    if scoreList[0] == False:
        print("Already Scored")
        return False
    scoreList[1] = lower_Points(category)
    scoreList[0] = False
    print("Scored", scoreList[1], "points in the", displayName, "category")
    return True
    
    
def score_Upper(category):
    scoreList, diceName, pointsPerDie = upper_Categories[category]
    if scoreList[0] == False:
        print("Already Scored")
        return False
    scoreList[1] = counted_Dice[diceName] * pointsPerDie
    scoreList[0] = False
    print("Scored", scoreList[1], "points in the", diceName, "category")
    return True

def update_Totals():
    global upperScore, bonusScore, upperTotal, lowerTotal
    upperScore = onesScore[1] + twosScore[1] + threesScore[1] + foursScore[1] + fivesScore[1] + sixesScore[1]
    if upperScore >= 63:
        bonusScore = 35
    else:
        bonusScore = 0
    upperTotal = upperScore + bonusScore
    lowerTotal = threeOAKScore[1] + fourOAKScore[1] + fullHouseScore[1] + smallStraightScore[1] + largeStraightScore[1] + yahtzeeScore[1] + chanceScore[1] + (extraYahtzees * 100)

In [ ]:
#RUN THIS TO PLAY THE GAME
initialize_Game()
for i in range(13):
    print("\n----- Turn", i + 1, "of 13 -----")
    do_Turn()
displayBoard()
print("\n Game over!")

In [26]:
#Run to view scorecard (does not affect or reset anything)
update_Totals()
display_Board()


| Ones               |     0 |
-------------------------------
| Twos               |     0 |
-------------------------------
| Threes             |     3 |
-------------------------------
| Fours              |     0 |
-------------------------------
| Fives              |     0 |
-------------------------------
| Sixes              |     0 |
| Total              |     3 |
-------------------------------
| Bonus (3/63)       |     0 |
-------------------------------
| Upper Total        |     3 |

| 3 of a Kind        |     0 |
-------------------------------
| 4 of a Kind        |     0 |
-------------------------------
| Full House         |     0 |
-------------------------------
| Sm. Straight (4)   |     0 |
-------------------------------
| Lg. Straight (5)   |     0 |
-------------------------------
| YAHTZEE            |     0 |
-------------------------------
| Chance             |     0 |
-------------------------------
| YAHTZEE BONUS      |     0 |
| Upper Total        |  